# 📊 Project 14.1 — Sensor Stream Analyser
**DSA for Mechatronics · Week 14 — Review & Final Project**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.

**Topics integrated:** Two Pointers (W03) · Sliding Window (W03) · Hash Maps (W05) · Binary Search (W11)


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import defaultdict, deque, Counter, OrderedDict
from functools import lru_cache
import bisect

_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A telemetry system streams integer sensor readings in real time.
We apply four classic array/stream algorithms to extract insights:

1. **Two-pointer pair sum** — find all pairs in a sorted reading log that sum to a target voltage (two pointers on sorted array)
2. **Three-sum triplets** — find all unique triplets that sum to zero — a signal calibration check (sort + fix-one + two pointers)
3. **Sliding window maximum** — the peak reading over any window of size *k* — used for anomaly detection (monotonic deque)
4. **Longest subarray without duplicate codes** — longest unbroken transmission segment with unique sensor codes (sliding window + hash map)


## Step 1 — Generate sensor datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_LOG      = random.randint(15, 22)
N_TRIPLETS = random.randint(14, 20)
N_WINDOW   = random.randint(12, 18)
K_WIN      = random.randint(3, 5)
N_CODES    = random.randint(14, 20)
CODES_ALPHA = "ABCDEFGHIJ"

PAIR_TARGET = random.randint(10, 25)

# Sorted reading log for pair-sum
READ_LOG   = sorted(random.randint(1, 20) for _ in range(N_LOG))

# Triplet array: seed with some that sum to zero
raw = [random.randint(-15, 15) for _ in range(N_TRIPLETS - 4)]
raw += [-raw[0]-raw[1], raw[0], raw[1], 0]   # guarantee ≥1 zero-sum triplet
random.shuffle(raw)
TRIPLET_ARR = raw

# Sliding window values
WIN_VALS   = [random.randint(1, 50) for _ in range(N_WINDOW)]

# Code stream (letters, some repeats)
CODE_STREAM = random.choices(CODES_ALPHA, k=N_CODES)

print(f"Sensor stream datasets:")
print(f"  Sorted read log  (n={N_LOG}) : {READ_LOG}")
print(f"  Pair-sum target             : {PAIR_TARGET}")
print(f"  Triplet array    (n={N_TRIPLETS}) : {TRIPLET_ARR}")
print(f"  Window values    (n={N_WINDOW}, k={K_WIN}) : {WIN_VALS}")
print(f"  Code stream      (n={N_CODES}) : {''.join(CODE_STREAM)}")


## Step 2 — Two-pointer pair sum on sorted log

In [ ]:
def two_pointer_pairs(arr, target):
    """
    Find all pairs in a SORTED array that sum to target.

    Two-pointer technique: place one pointer at the left end (smallest)
    and one at the right end (largest).
      - If sum < target → move left pointer right (increase sum).
      - If sum > target → move right pointer left (decrease sum).
      - If sum == target → record pair, advance both pointers.

    Skip duplicates by advancing past repeated values after each match.

    O(n) time (single pass), O(1) extra space.
    Works ONLY because the array is sorted — the key precondition.
    """
    left, right = 0, len(arr) - 1
    pairs = []
    steps = 0
    while left < right:
        steps += 1
        s = arr[left] + arr[right]
        if s == target:
            pairs.append((arr[left], arr[right]))
            # skip duplicates
            while left < right and arr[left]  == pairs[-1][0]: left  += 1
            while left < right and arr[right] == pairs[-1][1]: right -= 1
        elif s < target:
            left  += 1
        else:
            right -= 1
    return pairs, steps

pairs, pair_steps = two_pointer_pairs(READ_LOG, PAIR_TARGET)

print(f"Two-pointer pair sum (target={PAIR_TARGET}):")
print(f"  Sorted log : {READ_LOG}")
print()
print(f"  Steps taken   : {pair_steps}  (vs O(n²)={N_LOG**2} brute force)")
print(f"  Pairs found   : {len(pairs)}")
for a, b in pairs:
    print(f"    {a} + {b} = {PAIR_TARGET}  ✅")
# Brute-force verification
bf_pairs = sorted(set(
    (min(READ_LOG[i], READ_LOG[j]), max(READ_LOG[i], READ_LOG[j]))
    for i in range(N_LOG) for j in range(i+1, N_LOG)
    if READ_LOG[i] + READ_LOG[j] == PAIR_TARGET))
pairs_sorted = sorted(pairs)
print(f"  Brute-force  : {bf_pairs}")
print(f"  Match        : {'✅ YES' if pairs_sorted == bf_pairs else '❌ NO'}")


## Step 3 — Three-sum: find all unique zero-sum triplets

In [ ]:
def three_sum(nums):
    """
    Find all unique triplets (a, b, c) where a + b + c = 0.

    Algorithm:
      1. Sort the array.
      2. For each index i, use two pointers on nums[i+1:] to find
         pairs summing to -nums[i].
      3. Skip duplicate values of i, left, and right to avoid duplicate triplets.

    Why sorting first? It lets us:
      - Use two pointers (requires sorted sub-array).
      - Skip duplicates easily (equal values are adjacent).

    O(n²) time, O(1) extra space (not counting output).
    """
    nums_sorted = sorted(nums)
    n = len(nums_sorted)
    triplets = []
    for i in range(n - 2):
        # skip duplicate pivot
        if i > 0 and nums_sorted[i] == nums_sorted[i-1]:
            continue
        # Early exit: if smallest three are all positive, no solution possible
        if nums_sorted[i] > 0:
            break
        target_2 = -nums_sorted[i]
        left, right = i + 1, n - 1
        while left < right:
            s = nums_sorted[left] + nums_sorted[right]
            if s == target_2:
                triplets.append((nums_sorted[i], nums_sorted[left], nums_sorted[right]))
                while left < right and nums_sorted[left]  == nums_sorted[left+1]:  left  += 1
                while left < right and nums_sorted[right] == nums_sorted[right-1]: right -= 1
                left += 1; right -= 1
            elif s < target_2:
                left  += 1
            else:
                right -= 1
    return triplets, nums_sorted

triplets, ts = three_sum(TRIPLET_ARR)

print(f"Three-sum zero triplets (n={N_TRIPLETS}):")
print(f"  Input (sorted) : {ts}")
print()
print(f"  Triplets found : {len(triplets)}")
for a, b, c in triplets:
    print(f"    {a:4d} + {b:4d} + {c:4d} = {a+b+c}  ✅")
# Verify all sum to 0 and are unique
all_zero = all(a+b+c == 0 for a, b, c in triplets)
unique   = len(triplets) == len(set(triplets))
print(f"  All sum to 0   : {'✅ YES' if all_zero else '❌ NO'}")
print(f"  All unique     : {'✅ YES' if unique else '❌ NO'}")


## Step 4 — Sliding window maximum (monotonic deque)

In [ ]:
def sliding_window_max(nums, k):
    """
    Maximum value in every sliding window of size k.

    Naive approach: O(n·k) — for each window, scan all k elements.
    Monotonic deque: O(n) — maintain a deque of indices whose
    corresponding values are in decreasing order.

    Key invariant: deque stores indices of CANDIDATE maximum values.
      - The front is always the index of the current window's maximum.
      - When the front index falls outside the window, pop it.
      - Before adding a new index, pop from the BACK any index whose
        value is ≤ the new value (those can never be the max again).

    Result: front of deque is always the argmax for the current window.
    O(n) total — each index is pushed and popped at most once.
    """
    dq  = deque()   # stores indices, decreasing values
    res = []
    for i, val in enumerate(nums):
        # Remove indices outside this window
        while dq and dq[0] <= i - k:
            dq.popleft()
        # Maintain decreasing order in deque
        while dq and nums[dq[-1]] <= val:
            dq.pop()
        dq.append(i)
        # Start recording once first full window is filled
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

win_maxes = sliding_window_max(WIN_VALS, K_WIN)
# Naive verification
naive = [max(WIN_VALS[i:i+K_WIN]) for i in range(len(WIN_VALS) - K_WIN + 1)]

print(f"Sliding window maximum (n={N_WINDOW}, k={K_WIN}):")
print(f"  Values   : {WIN_VALS}")
print()
print(f"  Window maxima ({len(win_maxes)} windows):")
for i, m in enumerate(win_maxes):
    window = WIN_VALS[i:i+K_WIN]
    print(f"    window[{i}:{i+K_WIN}] = {window} → max = {m}")
print()
print(f"  Deque result  : {win_maxes}")
print(f"  Naive result  : {naive}")
print(f"  Match         : {'✅ YES' if win_maxes == naive else '❌ NO'}")
print(f"  Overall max   : {max(WIN_VALS)}")
print(f"  Max of maxima : {max(win_maxes)}")


## Step 5 — Longest subarray without duplicate codes

In [ ]:
def longest_no_dup(stream):
    """
    Longest contiguous subarray with no duplicate elements.

    Sliding window + hash map:
      - left, right = window boundaries.
      - last_seen[char] = most recent index where char appeared.
      - When a duplicate is found inside the window:
        move left to last_seen[char] + 1 (jump past the duplicate).
      - Update last_seen[char] = right always.
      - Track maximum window length.

    O(n) time, O(alphabet_size) space.
    Generalises to any hashable elements, not just characters.
    """
    last_seen = {}
    left = 0
    best_len   = 0
    best_start = 0
    for right, ch in enumerate(stream):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1
        last_seen[ch] = right
        if right - left + 1 > best_len:
            best_len   = right - left + 1
            best_start = left
    return best_len, best_start, stream[best_start:best_start+best_len]

lnd_len, lnd_start, lnd_sub = longest_no_dup(CODE_STREAM)

print(f"Longest subarray without duplicate codes (n={N_CODES}):")
print(f"  Code stream : {''.join(CODE_STREAM)}")
print()
print(f"  Best window   : indices [{lnd_start}, {lnd_start+lnd_len-1}]")
print(f"  Substring     : {''.join(lnd_sub)}  (length={lnd_len})")
print(f"  No duplicates : {'✅ YES' if len(lnd_sub) == len(set(lnd_sub)) else '❌ NO'}")
print()
# Show all window lengths for context
print(f"  Window trace (first 10 positions):")
last_seen_t = {}; left_t = 0
for right in range(min(10, N_CODES)):
    ch = CODE_STREAM[right]
    if ch in last_seen_t and last_seen_t[ch] >= left_t:
        left_t = last_seen_t[ch] + 1
    last_seen_t[ch] = right
    wlen = right - left_t + 1
    print(f"    i={right} char={ch}  window=['{''.join(CODE_STREAM[left_t:right+1])}']  len={wlen}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " SENSOR STREAM ANALYSER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TWO-POINTER PAIR SUM" + " "*(W-22) + "║")
print(f"║  {'Log size (n)':<28}: {N_LOG:<26}║")
print(f"║  {'Target':<28}: {PAIR_TARGET:<26}║")
print(f"║  {'Pairs found':<28}: {len(pairs):<26}║")
print(f"║  {'Pointer steps':<28}: {pair_steps} (vs {N_NODES**2} brute force){'':<6}║".replace("NODES", "LOG") if False else
      f"║  {'Pointer steps':<28}: {pair_steps:<26}║")
print(f"║  {'Verified correct':<28}: {'YES ✅' if pairs_sorted == bf_pairs else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  THREE-SUM TRIPLETS" + " "*(W-20) + "║")
print(f"║  {'Array size (n)':<28}: {N_TRIPLETS:<26}║")
print(f"║  {'Triplets found':<28}: {len(triplets):<26}║")
print(f"║  {'All sum to zero':<28}: {'YES ✅' if all_zero else 'NO ❌':<26}║")
print(f"║  {'All unique':<28}: {'YES ✅' if unique else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  SLIDING WINDOW MAXIMUM" + " "*(W-24) + "║")
print(f"║  {'Sequence length (n)':<28}: {N_WINDOW:<26}║")
print(f"║  {'Window size (k)':<28}: {K_WIN:<26}║")
print(f"║  {'Windows computed':<28}: {len(win_maxes):<26}║")
print(f"║  {'Max of all maxima':<28}: {max(win_maxes):<26}║")
print(f"║  {'Match naive O(nk)':<28}: {'YES ✅' if win_maxes == naive else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  LONGEST NO-DUPLICATE WINDOW" + " "*(W-29) + "║")
print(f"║  {'Stream length (n)':<28}: {N_CODES:<26}║")
print(f"║  {'Longest window':<28}: {lnd_len:<26}║")
print(f"║  {'Window content':<28}: {''.join(lnd_sub):<26}║")
print(f"║  {'No duplicates':<28}: {'YES ✅' if len(lnd_sub)==len(set(lnd_sub)) else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What do they tell you about the algorithms in this project?

*Your answer here:*

---

**Q2 — Which technique was most surprising, and why?**
Pick one algorithm from this project. Explain why the approach works — and give one scenario where it would *fail* or need modification.

*Your answer here:*

---

**Q3 — How does this project connect to earlier weeks?**
Name at least two specific weeks whose topics appear in this project and explain the connection.

*Your answer here:*
